In [ ]:
import os, gc, time, numpy as np, pandas as pd, torch, timm, torch.nn as nn
import matplotlib.pyplot as plt
from scipy.stats import rankdata
from scipy.optimize import minimize_scalar
from sklearn.linear_model import LogisticRegression
from tqdm import tqdm

LABELS = ["Cardiomegaly", "Edema", "Consolidation", "Atelectasis", "Pneumothorax", "Pleural Effusion"]
L = len(LABELS)
EPOCHS, SEEDS, BS, LR = 7, [0, 1, 2], 64, 1e-4          # identical to your DenseNet runs
OUT = r"G:\runs"; XD = rf"{OUT}\part34"; os.makedirs(XD, exist_ok=True)
ARCH  = {"DenseNet121": "densenet121", "ConvNeXt-T": "convnext_tiny.fb_in1k"}  # fb_in1k = ImageNet-1k weights, same as DenseNet
SHORT = {"DenseNet121": "densenet121", "ConvNeXt-T": "convnext_tiny"}

plt.rcParams.update({"font.family": "serif", "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix", "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7, "axes.linewidth": 0.6,
    "lines.linewidth": 1.0, "grid.linewidth": 0.4, "figure.dpi": 150, "pdf.fonttype": 42})
COL1, COL2 = 3.5, 7.16
def save_fig(fig, name):
    fig.savefig(rf"{XD}\{name}.pdf", bbox_inches="tight"); fig.savefig(rf"{XD}\{name}.png", dpi=600, bbox_inches="tight"); plt.show()

# ---- your tables and caches ----
nih = pd.read_csv(r"G:\nih_split.csv")
tr_i = np.where(nih.split_rev.values == "train")[0]; va_i = np.where(nih.split_rev.values == "val")[0]
te_i = np.where(nih.split_rev.values == "test")[0]
off = nih.official.values == "test"; assert (te_i == np.where(off)[0]).all()
cx = pd.read_csv(r"G:\chexpert_final_clean.csv"); cxt = cx[cx.split == "test"].reset_index(drop=True)
cxm = cx[cx.split.isin(["train", "val"])].reset_index(drop=True)
cx_tr = np.where(cxm.split.values == "train")[0]; cx_va = np.where(cxm.split.values == "val")[0]
assert os.path.exists(r"G:\cx_trainval_224.u8"), "run the 'caching CheXpert train+val' cell of che44-nih77 first"
arr = np.memmap(r"G:\nih_224.u8", dtype=np.uint8, mode="r", shape=(len(nih), 224, 224))
CXA = np.memmap(r"G:\cx_trainval_224.u8", dtype=np.uint8, mode="r", shape=(len(cxm), 224, 224))
XC  = np.load(r"G:\cx_test_224.npy")
Yn = nih[LABELS].values.astype(int); Ynt = Yn[off]
Ycm = cxm[LABELS].values.astype(int); Yc = cxt[LABELS].values.astype(int)

MEAN = torch.tensor([0.485, 0.456, 0.406], device="cuda").view(1, 3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225], device="cuda").view(1, 3, 1, 1)
def gpu_batch(A, idx, flip=False):
    x = torch.from_numpy(np.array(A[idx])).cuda(non_blocking=True).float().div(255).unsqueeze(1)
    if flip:
        f = torch.rand(len(idx), device="cuda") < 0.5
        x = torch.where(f.view(-1, 1, 1, 1), x.flip(-1), x)
    return (x.repeat(1, 3, 1, 1) - MEAN) / STD
@torch.no_grad()
def predict_idx(m, A, idx, bs=256):
    m.eval(); out = []
    for i in range(0, len(idx), bs):
        with torch.autocast("cuda"):
            o = m(gpu_batch(A, idx[i:i + bs]))
        out.append(torch.sigmoid(o.float()).cpu())
    return torch.cat(out).numpy()
def fauc(y, s):
    y = np.asarray(y).astype(bool); r = rankdata(s); n1 = y.sum(); n0 = len(y) - n1
    return (r[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)
def mean_auc(Y, P): return float(np.mean([fauc(Y[:, i], P[:, i]) for i in range(L)]))
def bce(Y, P):
    P = np.clip(P, 1e-6, 1 - 1e-6); return float(-(Y * np.log(P) + (1 - Y) * np.log(1 - P)).mean())
def groups(p): return list(pd.Series(np.arange(len(p))).groupby(np.asarray(p)).apply(np.array).values)
print("NIH train/val/test:", len(tr_i), len(va_i), len(te_i), "| CheXpert train/val/test:", len(cx_tr), len(cx_va), len(cxt))

In [ ]:
lossf = nn.BCEWithLogitsLoss()
SRC = {"nih": dict(A=arr, Y=Yn,  tr=tr_i,  va=va_i),
       "cx":  dict(A=CXA, Y=Ycm, tr=cx_tr, va=cx_va)}
def tagpath(a, src, s): return rf"{XD}\{SHORT[a]}_{src}_e{EPOCHS}_seed{s}"

def train_one(a, src, seed):
    tag = tagpath(a, src, seed)
    BEST, CKPT, HIST, PRED = tag + ".pt", tag + "_last.pt", tag + "_history.csv", tag + "_preds.npz"
    if os.path.exists(PRED): print("skip", tag); return
    A, Y, tr_idx, va_idx = (SRC[src][k] for k in ("A", "Y", "tr", "va"))
    torch.manual_seed(seed); np.random.seed(seed)
    m = timm.create_model(ARCH[a], pretrained=True, num_classes=L).cuda()
    o = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=1e-4)
    spe = int(np.ceil(len(tr_idx) / BS))
    sc = torch.optim.lr_scheduler.CosineAnnealingLR(o, T_max=EPOCHS * spe)
    sl = torch.amp.GradScaler("cuda"); hist, best, start = [], 0.0, 0
    if os.path.exists(CKPT):
        ck = torch.load(CKPT, map_location="cuda", weights_only=False)
        m.load_state_dict(ck["model"]); o.load_state_dict(ck["opt"]); sc.load_state_dict(ck["sched"])
        sl.load_state_dict(ck["scaler"]); hist, best, start = ck["hist"], ck["best"], ck["ep"] + 1
        print("resumed, next epoch", start + 1)
    for ep in range(start, EPOCHS):
        perm = np.random.default_rng(seed * 100 + ep).permutation(tr_idx); m.train(); t0 = time.time()
        rl, n = torch.zeros((), device="cuda"), 0
        for k in tqdm(range(spe), desc=f"{a} {src} seed {seed} ep {ep+1}/{EPOCHS}"):
            idx = np.sort(perm[k * BS:(k + 1) * BS])
            x = gpu_batch(A, idx, flip=True); y = torch.from_numpy(Y[idx]).cuda().float()
            o.zero_grad(set_to_none=True)
            with torch.autocast("cuda"):
                loss = lossf(m(x), y)
            sl.scale(loss).backward(); sl.step(o); sl.update(); sc.step()
            rl += loss.detach() * len(idx); n += len(idx)
        Pv = predict_idx(m, A, va_idx); Yv = Y[va_idx]
        row = dict(epoch=ep + 1, train_loss=float(rl) / n, val_loss=bce(Yv, Pv), val_auroc=mean_auc(Yv, Pv), sec=time.time() - t0)
        hist.append(row); pd.DataFrame(hist).to_csv(HIST, index=False)
        print(f"{a} {src} seed {seed} ep {ep+1} | loss {row['train_loss']:.4f}/{row['val_loss']:.4f} | val AUROC {row['val_auroc']:.4f} | {row['sec']/60:.1f} min")
        if row["val_auroc"] > best: best = row["val_auroc"]; torch.save(m.state_dict(), BEST)
        torch.save(dict(model=m.state_dict(), opt=o.state_dict(), sched=sc.state_dict(), scaler=sl.state_dict(), hist=hist, best=best, ep=ep), CKPT)
    m.load_state_dict(torch.load(BEST, map_location="cuda", weights_only=True))
    z = dict(val_P=predict_idx(m, A, va_idx), val_Y=Y[va_idx],
             cx_test_P=predict_idx(m, XC, np.arange(len(XC))), cx_test_Y=Yc,
             nih_test_P=predict_idx(m, arr, te_i), nih_test_Y=Ynt)
    np.savez(PRED, **z); del m, o, sc, sl; gc.collect(); torch.cuda.empty_cache()

for src in ["nih", "cx"]:
    for s in SEEDS:                 # tip: test the pipeline first with SEEDS_RUN = [0]
        train_one("ConvNeXt-T", src, s)

In [ ]:
ZF  = {s: np.load(rf"{OUT}\densenet121_e{EPOCHS}_seed{s}_results\preds.npz") for s in SEEDS}      # DenseNet, CheXpert-trained
NPF = {s: np.load(rf"{OUT}\nih_eval\seed{s}_nih_preds.npy") for s in SEEDS}                       # DenseNet, CheXpert-trained, all NIH
ZR  = {s: np.load(rf"{OUT}\nih_train\densenet121_nih_e{EPOCHS}_seed{s}_preds.npz") for s in SEEDS} # DenseNet, NIH-trained
XF  = {s: np.load(tagpath("ConvNeXt-T", "cx",  s) + "_preds.npz") for s in SEEDS}                  # ConvNeXt, CheXpert-trained
XR  = {s: np.load(tagpath("ConvNeXt-T", "nih", s) + "_preds.npz") for s in SEEDS}                  # ConvNeXt, NIH-trained
assert (ZF[0]["test_Y"] == Yc).all() and (ZR[0]["nih_test_Y"] == Ynt).all()
assert (XF[0]["cx_test_Y"] == Yc).all() and (XR[0]["nih_test_Y"] == Ynt).all()

DIRS = ["CheXpert→CheXpert", "CheXpert→NIH", "NIH→NIH", "NIH→CheXpert"]
PR = {"DenseNet121": {"CheXpert→CheXpert": (Yc,  {s: ZF[s]["test_P"] for s in SEEDS}),
                      "CheXpert→NIH":      (Ynt, {s: NPF[s][off] for s in SEEDS}),
                      "NIH→NIH":           (Ynt, {s: ZR[s]["nih_test_P"] for s in SEEDS}),
                      "NIH→CheXpert":      (Yc,  {s: ZR[s]["cx_test_P"] for s in SEEDS})},
      "ConvNeXt-T":  {"CheXpert→CheXpert": (Yc,  {s: XF[s]["cx_test_P"] for s in SEEDS}),
                      "CheXpert→NIH":      (Ynt, {s: XF[s]["nih_test_P"] for s in SEEDS}),
                      "NIH→NIH":           (Ynt, {s: XR[s]["nih_test_P"] for s in SEEDS}),
                      "NIH→CheXpert":      (Yc,  {s: XR[s]["cx_test_P"] for s in SEEDS})}}

pmf = lambda x: f"{np.mean(x):.3f} ± {np.std(x, ddof=1):.3f}"
seedm = lambda a, d: [mean_auc(PR[a][d][0], PR[a][d][1][s]) for s in SEEDS]
mat = pd.DataFrame({d: {a: pmf(seedm(a, d)) for a in PR} for d in DIRS}); mat.index.name = "Mean AUROC"
display(mat); mat.to_csv(rf"{XD}\table_arch_transfer_matrix.csv")

In [ ]:
B = 300; rng = np.random.default_rng(0)
gc_, gn_ = groups(cxt.patient_id.values), groups(nih.patient_id.values[off])
USE_C = {"CheXpert→CheXpert", "NIH→CheXpert"}                      # directions evaluated on the CheXpert test set
lab_auc = lambda Y, Ps, ix: np.array([np.mean([fauc(Y[ix, i], Ps[s][ix, i]) for s in SEEDS]) for i in range(L)])

PT = {a: {d: lab_auc(Y, Ps, np.arange(len(Y))) for d, (Y, Ps) in PR[a].items()} for a in PR}
BSX = {a: {d: np.zeros((B, L)) for d in DIRS} for a in PR}
for r_ in tqdm(range(B), desc="bootstrap"):
    ic = np.concatenate([gc_[k] for k in rng.integers(0, len(gc_), len(gc_))])
    iN = np.concatenate([gn_[k] for k in rng.integers(0, len(gn_), len(gn_))])
    for a in PR:                                                   # same resample for both architectures = paired
        for d in DIRS:
            Y, Ps = PR[a][d]; BSX[a][d][r_] = lab_auc(Y, Ps, ic if d in USE_C else iN)

def gaps(src):  # src is PT or BSX; returns forward gap, reverse gap, asymmetry per architecture
    return {a: dict(fw=src[a]["CheXpert→CheXpert"] - src[a]["CheXpert→NIH"],
                    rv=src[a]["NIH→NIH"] - src[a]["NIH→CheXpert"]) for a in PR}
gp, gb = gaps(PT), gaps(BSX)
for a in PR:
    gp[a]["asym"] = gp[a]["rv"] - gp[a]["fw"]; gb[a]["asym"] = gb[a]["rv"] - gb[a]["fw"]

NAMES = {"fw": "Forward gap (CheXpert→NIH)", "rv": "Reverse gap (NIH→CheXpert)", "asym": "Asymmetry (reverse − forward)"}
rows = []
for a in PR:
    for k, nm in NAMES.items():
        lo, hi = np.percentile(gb[a][k].mean(1), [2.5, 97.5])
        rows.append({"Model": a, "Quantity": nm, "Mean over labels": gp[a][k].mean(), "95% CI": f"[{lo:+.3f}, {hi:+.3f}]"})
d_as = gb["ConvNeXt-T"]["asym"].mean(1) - gb["DenseNet121"]["asym"].mean(1)
lo, hi = np.percentile(d_as, [2.5, 97.5])
rows.append({"Model": "ConvNeXt-T − DenseNet121", "Quantity": "Difference in asymmetry",
             "Mean over labels": gp["ConvNeXt-T"]["asym"].mean() - gp["DenseNet121"]["asym"].mean(), "95% CI": f"[{lo:+.3f}, {hi:+.3f}]"})
cmp_tab = pd.DataFrame(rows); display(cmp_tab.round(3)); cmp_tab.to_csv(rf"{XD}\table_arch_gaps.csv", index=False)

pl = pd.DataFrame({f"{a} {k}": gp[a][k] for a in PR for k in ["fw", "rv"]}, index=LABELS); display(pl.round(3))
pl.to_csv(rf"{XD}\table_arch_gaps_per_label.csv")

In [ ]:
clip = lambda a: np.clip(a, 0, None); ci = lambda a: np.percentile(a, [2.5, 97.5], axis=0); x = np.arange(L)

# Fig 3A: transfer matrices side by side
V = {a: np.array([[PT[a]["CheXpert→CheXpert"].mean(), PT[a]["CheXpert→NIH"].mean()],
                  [PT[a]["NIH→CheXpert"].mean(),      PT[a]["NIH→NIH"].mean()]]) for a in PR}
vmin, vmax = min(v.min() for v in V.values()) - 0.02, max(v.max() for v in V.values()) + 0.02
fig, ax = plt.subplots(1, 2, figsize=(COL2, 2.5))
for a_, a in zip(ax, PR):
    a_.imshow(V[a], cmap="Greys", vmin=vmin, vmax=vmax)
    for i in range(2):
        for j in range(2):
            a_.text(j, i, f"{V[a][i, j]:.3f}", ha="center", va="center", color="white" if V[a][i, j] > (vmin + vmax) / 2 else "black")
    a_.set_xticks([0, 1]); a_.set_xticklabels(["CheXpert test", "NIH test"]); a_.set_yticks([0, 1]); a_.set_yticklabels(["CheXpert", "NIH"])
    a_.set(title=a, xlabel="Tested on", ylabel="Trained on"); a_.grid(False)
fig.tight_layout(); save_fig(fig, "fig_arch_transfer_matrices")

# Fig 3B: per-label gaps, both directions, both architectures
fig, ax = plt.subplots(1, 2, figsize=(COL2, 2.8), sharey=True)
for a_, a in zip(ax, PR):
    cf, cr = ci(gb[a]["fw"]), ci(gb[a]["rv"]); fw, rv = gp[a]["fw"], gp[a]["rv"]
    a_.bar(x - 0.2, fw, 0.4, yerr=[clip(fw - cf[0]), clip(cf[1] - fw)], capsize=1.5, color="0.35", label="CheXpert → NIH")
    a_.bar(x + 0.2, rv, 0.4, yerr=[clip(rv - cr[0]), clip(cr[1] - rv)], capsize=1.5, color="0.75", edgecolor="k", lw=0.4, label="NIH → CheXpert")
    a_.axhline(0, color="k", lw=0.6); a_.set_xticks(x); a_.set_xticklabels(LABELS, rotation=35, ha="right")
    a_.set(title=a); a_.grid(True, axis="y", alpha=0.3)
ax[0].set_ylabel("AUROC drop (in-hospital − external)"); ax[0].legend(frameon=False)
fig.tight_layout(); save_fig(fig, "fig_arch_gaps_per_label")

# Fig 3C: mean gaps and asymmetry with CIs
fig, ax = plt.subplots(figsize=(COL1 * 1.5, 2.7)); w = 0.36
for k, (a, sh) in enumerate(zip(PR, ["0.3", "0.7"])):
    pt = np.array([gp[a][q].mean() for q in NAMES]); c = np.array([np.percentile(gb[a][q].mean(1), [2.5, 97.5]) for q in NAMES])
    ax.bar(np.arange(3) + (k - 0.5) * w, pt, w, yerr=[clip(pt - c[:, 0]), clip(c[:, 1] - pt)], capsize=2, color=sh, edgecolor="k", lw=0.4, label=a)
ax.set_xticks(range(3)); ax.set_xticklabels(["Forward gap", "Reverse gap", "Asymmetry"]); ax.axhline(0, color="k", lw=0.6)
ax.set(ylabel="Mean AUROC difference"); ax.legend(frameon=False); ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout(); save_fig(fig, "fig_arch_summary")

In [ ]:
CK = {("DenseNet121", "cx"):  lambda s: rf"{OUT}\densenet121_e{EPOCHS}_seed{s}.pt",
      ("DenseNet121", "nih"): lambda s: rf"{OUT}\nih_train\densenet121_nih_e{EPOCHS}_seed{s}.pt",
      ("ConvNeXt-T",  "cx"):  lambda s: tagpath("ConvNeXt-T", "cx", s) + ".pt",
      ("ConvNeXt-T",  "nih"): lambda s: tagpath("ConvNeXt-T", "nih", s) + ".pt"}
pool_path = lambda a, src, s: tagpath(a, src, s) + "_pool.npz"

for (a, src), f in CK.items():
    for s in SEEDS:
        if os.path.exists(pool_path(a, src, s)): continue
        m = timm.create_model(ARCH[a], pretrained=False, num_classes=L).cuda()
        m.load_state_dict(torch.load(f(s), map_location="cuda", weights_only=True))
        A_, idx_ = (CXA, cx_va) if src == "nih" else (arr, va_i)      # target hospital's validation split
        np.savez(pool_path(a, src, s), pool_P=predict_idx(m, A_, idx_))
        del m; gc.collect(); torch.cuda.empty_cache(); print("pool done:", a, src, s)

def get_inputs(a, src, s):
    """source-val, in-hospital test, external test, target pool: each (Y, P)"""
    if a == "DenseNet121":
        if src == "cx":  z = ZF[s]; val = (z["val_Y"].astype(int), z["val_P"]); inn = (Yc, z["test_P"]); ext = (Ynt, NPF[s][off])
        else:            z = ZR[s]; val = (z["nih_val_Y"].astype(int), z["nih_val_P"]); inn = (Ynt, z["nih_test_P"]); ext = (Yc, z["cx_test_P"])
    else:
        z = XF[s] if src == "cx" else XR[s]; val = (z["val_Y"].astype(int), z["val_P"])
        inn, ext = ((Yc, z["cx_test_P"]), (Ynt, z["nih_test_P"])) if src == "cx" else ((Ynt, z["nih_test_P"]), (Yc, z["cx_test_P"]))
    pool_Y = Yn[va_i] if src == "cx" else Ycm[cx_va]
    return val, inn, ext, (pool_Y, np.load(pool_path(a, src, s))["pool_P"])

In [ ]:
GRID = np.linspace(0.02, 0.95, 94)
sig = lambda z: 1 / (1 + np.exp(-z))
def logit(p): p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))
def ece_of(y, p, nb=10):                                   # same definition as your ECE tables
    bi = np.clip((p * nb).astype(int), 0, nb - 1); e = 0.0
    for k in range(nb):
        mk = bi == k
        if mk.any(): e += mk.mean() * abs(p[mk].mean() - y[mk].mean())
    return e
def f1_at(y, p, t):
    pr = p >= t; y1 = y == 1; tp = (pr & y1).sum(); d = 2 * tp + (pr & ~y1).sum() + (~pr & y1).sum()
    return 2 * tp / d if d else 0.0
def best_thr(y, p):                                        # vectorised max-F1 threshold
    pr = p[:, None] >= GRID[None, :]; y1 = (y == 1)[:, None]
    tp = (pr & y1).sum(0); fp = pr.sum(0) - tp; fn = y1.sum() - tp
    return GRID[np.argmax(2 * tp / np.maximum(2 * tp + fp + fn, 1))]
def fit_bias(z, y):                                        # 1 parameter: shift the logit
    f = lambda b: np.mean(np.logaddexp(0, z + b) - y * (z + b))
    return minimize_scalar(f, bounds=(-8, 8), method="bounded").x
def fit_platt(z, y):                                       # 2 parameters; slope clipped to stay positive (keeps ranking)
    lr = LogisticRegression(C=1e6, max_iter=500).fit(z.reshape(-1, 1), y)
    return float(np.clip(lr.coef_[0, 0], 0.1, 5.0)), float(lr.intercept_[0])

METHODS = ["Threshold only", "Bias (intercept)", "Platt (slope+intercept)"]
def apply_method(method, zp, yp, ze, thr0):
    """fit on the k pool samples of ONE label; return corrected external probabilities, threshold, and whether a fit happened"""
    a, b, t, fit = 1.0, 0.0, thr0, False
    npos = int(yp.sum()); nneg = len(yp) - npos
    if method == "Threshold only":
        if npos >= 3 and nneg >= 3: t = best_thr(yp, sig(zp)); fit = True
    elif npos >= 5 and nneg >= 5:                          # too few positives → fall back to the uncorrected model
        if method.startswith("Bias"): b = fit_bias(zp, yp)
        else: a, b = fit_platt(zp, yp)
        t = best_thr(yp, sig(a * zp + b)); fit = True
    return sig(a * ze + b), t, fit

In [ ]:
KS, DRAWS = [0, 25, 50, 100, 250, 500, 1000, 2500, 5000], 20
DIRN = {"nih": "NIH→CheXpert", "cx": "CheXpert→NIH"}
rows, ref = [], []
for a in ARCH:
    for src in ["nih", "cx"]:
        for s in tqdm(SEEDS, desc=f"{a} {DIRN[src]}"):
            val, inn, ext, pool = get_inputs(a, src, s)
            (Ye, Pe), (Yp, Pp), (Yi, Pi) = ext, pool, inn; Ze, Zp = logit(Pe), logit(Pp)
            thr0 = np.array([best_thr(val[0][:, j], val[1][:, j]) for j in range(L)])
            ref.append(dict(arch=a, src=src, seed=s, auroc_in=mean_auc(Yi, Pi), auroc_ext=mean_auc(Ye, Pe),
                            ece_in=np.mean([ece_of(Yi[:, j], Pi[:, j]) for j in range(L)]),
                            f1_in=np.mean([f1_at(Yi[:, j], Pi[:, j], thr0[j]) for j in range(L)])))
            for K in KS:
                for dr in range(1 if K == 0 else DRAWS):
                    idx = np.random.default_rng([s, K, dr]).choice(len(Yp), K, replace=False) if K else np.array([], int)
                    for meth in METHODS:                   # same subset for all methods = paired comparison
                        es, fs, nf = [], [], 0
                        for j in range(L):
                            pe, t, fit = apply_method(meth, Zp[idx, j], Yp[idx, j], Ze[:, j], thr0[j])
                            es.append(ece_of(Ye[:, j], pe)); fs.append(f1_at(Ye[:, j], pe, t)); nf += fit
                        rows.append(dict(arch=a, src=src, seed=s, method=meth, K=K, draw=dr, ece=np.mean(es), f1=np.mean(fs), n_fit=nf))
R, REF = pd.DataFrame(rows), pd.DataFrame(ref)
R.to_csv(rf"{XD}\recal_raw.csv", index=False); REF.to_csv(rf"{XD}\recal_reference.csv", index=False)

per_seed = R.groupby(["arch", "src", "method", "K", "seed"])[["ece", "f1", "n_fit"]].mean().reset_index()
per_seed["direction"] = per_seed.src.map(DIRN)
b0 = per_seed[per_seed.K == 0].set_index(["arch", "src", "method", "seed"])[["ece", "f1"]]
ps = (per_seed.join(b0, on=["arch", "src", "method", "seed"], rsuffix="_0")
              .join(REF.set_index(["arch", "src", "seed"])[["ece_in", "f1_in"]], on=["arch", "src", "seed"]))
safe = lambda num, den: np.where(np.abs(den) > 0.01, num / den, np.nan)       # % recovered only means something when the gap is large
ps["rec_f1"]  = safe(ps.f1 - ps.f1_0, ps.f1_in - ps.f1_0)
ps["rec_ece"] = safe(ps.ece_0 - ps.ece, ps.ece_0 - ps.ece_in)
refm = REF.groupby(["arch", "src"])[["ece_in", "f1_in"]].mean()

ms = lambda x: f"{x.mean():.3f} ± {x.std(ddof=1):.3f}"
for met in ["ece", "f1"]:
    t = per_seed[per_seed.K.isin([0, 100, 500, 2500])].groupby(["arch", "direction", "method", "K"])[met].agg(ms).unstack("K")
    print(f"\n=== {met.upper()} on the external test set (mean ± std over seeds) ==="); display(t); t.to_csv(rf"{XD}\table_recal_{met}.csv")
rt = ps[ps.K.isin([100, 500, 2500])].groupby(["arch", "direction", "method", "K"])[["rec_f1", "rec_ece"]].agg(lambda x: f"{100*x.mean():.0f}% ± {100*x.std(ddof=1):.0f}").unstack("K")
print("\n=== % of the in-hospital gap recovered ==="); display(rt); rt.to_csv(rf"{XD}\table_recal_recovery.csv")
print("\nAUROC is identical for all methods (monotone transform):"); display(REF.groupby(["arch", "src"])[["auroc_in", "auroc_ext"]].mean().round(3))
print("\nLabels actually fitted (of 6) at each k, Platt:"); display(per_seed[per_seed.method.str.startswith("Platt")].groupby(["arch", "direction", "K"]).n_fit.mean().unstack("K").round(1))

In [ ]:
MC = {"Threshold only": "C0", "Bias (intercept)": "C2", "Platt (slope+intercept)": "C3"}
LS = {"DenseNet121": "-", "ConvNeXt-T": "--"}
sel = lambda a, src, m: per_seed[(per_seed.arch == a) & (per_seed.src == src) & (per_seed.method == m)].groupby("K")

# Fig 4A: ECE and F1 versus k, all methods, both architectures, both directions
fig, axes = plt.subplots(2, 2, figsize=(COL2, 5.2))
for r_, src in enumerate(["nih", "cx"]):
    for c_, (met, ttl) in enumerate([("ece", "ECE (lower is better)"), ("f1", "Macro-F1 (higher is better)")]):
        ax = axes[r_, c_]
        for a in ARCH:
            for m in METHODS:
                g = sel(a, src, m)[met].agg(["mean", "std"])
                ax.plot(g.index, g["mean"], LS[a], color=MC[m], marker="o", ms=2.5, label=f"{a}, {m}")
                ax.fill_between(g.index, g["mean"] - g["std"], g["mean"] + g["std"], color=MC[m], alpha=0.12, lw=0)
            ax.axhline(refm.loc[(a, src), met + "_in"], color="k", ls=LS[a], lw=0.6, alpha=0.6)   # in-hospital reference
        ax.set_xscale("symlog", linthresh=25); ax.grid(True, alpha=0.3)
        ax.set(xlabel="k labelled target images", title=f"{DIRN[src]}: {ttl}")
h, l_ = axes[0, 0].get_legend_handles_labels()
fig.legend(h, l_, loc="lower center", ncol=2, frameon=False, fontsize=6); fig.tight_layout(rect=[0, 0.1, 1, 1])
save_fig(fig, "fig_recal_vs_k")

# Fig 4B: % of gap recovered by Platt at k = 100, 500, 2500
KR = [100, 500, 2500]
rec = ps[ps.method.str.startswith("Platt") & ps.K.isin(KR)].groupby(["arch", "direction", "K"])[["rec_f1", "rec_ece"]].agg(["mean", "std"])
combos = [(a, d) for a in ARCH for d in DIRN.values()]
fig, ax = plt.subplots(1, 2, figsize=(COL2, 2.7)); w = 0.26
for a_, (col, ttl) in zip(ax, [("rec_f1", "F1 gap recovered"), ("rec_ece", "ECE gap recovered")]):
    for k, K in enumerate(KR):
        mu = [100 * rec.loc[(a, d, K), (col, "mean")] for a, d in combos]; sd = [100 * rec.loc[(a, d, K), (col, "std")] for a, d in combos]
        a_.bar(np.arange(len(combos)) + (k - 1) * w, mu, w, yerr=sd, capsize=1.5, color=["0.3", "0.55", "0.8"][k], edgecolor="k", lw=0.4, label=f"k = {K}")
    a_.axhline(100, color="k", ls=":", lw=0.6); a_.axhline(0, color="k", lw=0.6)
    a_.set_xticks(range(len(combos))); a_.set_xticklabels([f"{a}\n{d}" for a, d in combos], fontsize=5.5)
    a_.set(ylabel="% of in-hospital gap recovered", title=ttl); a_.grid(True, axis="y", alpha=0.3)
ax[0].legend(frameon=False, fontsize=6); fig.tight_layout(); save_fig(fig, "fig_recal_recovery")

# Fig 4C: reliability curves before / after Platt (DenseNet, NIH→CheXpert, seed 0, k = 500)
def reliab(y, p, nb=10):
    bi = np.clip((p * nb).astype(int), 0, nb - 1); xs, ys = [], []
    for k in range(nb):
        mk = bi == k
        if mk.sum() > 20: xs.append(p[mk].mean()); ys.append(y[mk].mean())
    return xs, ys
a0, src0, s0, K0 = "DenseNet121", "nih", 0, 500
val, inn, ext, pool = get_inputs(a0, src0, s0); (Ye, Pe), (Yp, Pp), (Yi, Pi) = ext, pool, inn
idx = np.random.default_rng([s0, K0, 0]).choice(len(Yp), K0, replace=False)
fig, axes = plt.subplots(2, 3, figsize=(COL2, 4.3))
for j, (ax, l) in enumerate(zip(axes.ravel(), LABELS)):
    pc, _, _ = apply_method("Platt (slope+intercept)", logit(Pp)[idx, j], Yp[idx, j], logit(Pe[:, j]), 0.5)
    for nm, y_, p_, st in [("In-hospital (NIH test)", Yi[:, j], Pi[:, j], "-^"), ("External, uncorrected", Ye[:, j], Pe[:, j], "--s"),
                           (f"External, Platt k={K0}", Ye[:, j], pc, "-o")]:
        xs, ys = reliab(y_, p_); ax.plot(xs, ys, st, ms=2.5, label=nm)
    ax.plot([0, 1], [0, 1], "k:", lw=0.6); ax.set_title(l); ax.grid(True, alpha=0.3)
    ax.set(xlabel="Predicted probability", ylabel="Observed frequency")
axes[0, 0].legend(frameon=False, fontsize=5.5); fig.tight_layout(); save_fig(fig, "fig_recal_reliability")

In [ ]:
import pickle, shutil, json, datetime
import torch.nn.functional as F
from scipy.stats import mannwhitneyu

GD = rf"{XD}\gradcam"; os.makedirs(GD, exist_ok=True)
SHOW_LABELS = ["Pleural Effusion", "Edema"]      # shown inline; all other figures are only saved
NSTAT, KSHOW, GSEED = 200, 4, 11                 # images per group for statistics / shown per group / sampling seed
GROUPS = ["TP", "FN", "FP", "TN"]
GNAME = {"TP": "Correct positive (TP)", "FN": "Missed positive (FN)", "FP": "False alarm (FP)", "TN": "Correct negative (TN)"}
MRG = int(0.15 * 224)
EDGE = np.ones((224, 224), bool); EDGE[MRG:-MRG, MRG:-MRG] = False        # outer 15% frame
BOTTOM = np.zeros((224, 224), bool); BOTTOM[149:, :] = True                # bottom third

def load_model(a, src, seed=0):
    m = timm.create_model(ARCH[a], pretrained=False, num_classes=L).cuda()
    m.load_state_dict(torch.load(CK[(a, src)](seed), map_location="cuda", weights_only=True))
    return m.eval()

def gradcam_batch(model, imgs, cls, bs=50):
    """Grad-CAM on the last feature map; works for DenseNet121 and ConvNeXt-T (timm forward_features / forward_head)."""
    cams, probs = [], []
    for i in range(0, len(imgs), bs):
        x = torch.from_numpy(np.ascontiguousarray(imgs[i:i + bs])).cuda().float().div(255).unsqueeze(1).repeat(1, 3, 1, 1)
        x = (x - MEAN) / STD
        with torch.enable_grad():
            fm = model.forward_features(x)
            out = model.forward_head(fm)
            g = torch.autograd.grad(out[:, cls].sum(), fm)[0]       # images are independent in eval mode
        w = g.mean((2, 3), keepdim=True)
        cam = torch.relu((w * fm).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=(224, 224), mode="bilinear", align_corners=False)[:, 0]
        cam = cam / (cam.flatten(1).max(1).values.view(-1, 1, 1) + 1e-8)
        cams.append(cam.detach().cpu().numpy()); probs.append(torch.sigmoid(out[:, cls]).detach().cpu().numpy())
    return np.concatenate(cams), np.concatenate(probs)

def get_img(tname, i): return np.asarray(arr[te_i[i]]) if tname == "NIH" else XC[i]
def grp_ids(Y, P, j, t):
    y, p = Y[:, j], P[:, j]
    return {"TP": np.where((y == 1) & (p >= t))[0], "FN": np.where((y == 1) & (p < t))[0],
            "FP": np.where((y == 0) & (p >= t))[0], "TN": np.where((y == 0) & (p < t))[0]}

rng = np.random.default_rng(GSEED)
stat_rows, pick_rows, RAW, MEANCAM = [], [], {}, {}
for a in ARCH:
    os.makedirs(rf"{GD}\{SHORT[a]}", exist_ok=True)
    for src in ["nih", "cx"]:
        model = load_model(a, src, 0)
        val, inn, ext, _ = get_inputs(a, src, 0)
        thr0 = np.array([best_thr(val[0][:, j], val[1][:, j]) for j in range(L)])     # source-validation thresholds
        trained, away = ("NIH", "CheXpert") if src == "nih" else ("CheXpert", "NIH")
        for tk, (Y, P), tname in [("in", inn, trained), ("ext", ext, away)]:
            for j, l in enumerate(LABELS):
                ids = grp_ids(Y, P, j, thr0[j])
                fig, axs = plt.subplots(4, KSHOW, figsize=(COL2, 4.8))
                for r_, g in enumerate(GROUPS):
                    for c_ in range(KSHOW): axs[r_, c_].axis("off")
                    axs[r_, 0].set_title(f"{GNAME[g]}, n = {len(ids[g])}", loc="left", fontsize=6)
                    n = min(NSTAT, len(ids[g]))
                    row = dict(arch=a, src=src, trained_on=trained, tested_on=tname, setting="in-hospital" if tk == "in" else "external",
                               label=l, group=g, n_group=len(ids[g]), n_used=n, threshold=thr0[j])
                    if n == 0:
                        RAW[(a, src, tk, l, g)] = (np.array([]), np.array([]))
                        row.update(edge=np.nan, bottom=np.nan, empty=0); stat_rows.append(row); continue
                    pk = rng.choice(ids[g], n, replace=False)
                    cams, _ = gradcam_batch(model, np.stack([get_img(tname, i) for i in pk]), j)
                    valid = cams.max((1, 2)) > 0.5                                       # empty heatmap = all zeros
                    s_ = np.maximum(cams.sum((1, 2)), 1e-8)
                    ef = (cams[:, EDGE].sum(1) / s_)[valid]; bf = (cams[:, BOTTOM].sum(1) / s_)[valid]
                    RAW[(a, src, tk, l, g)] = (ef, bf)
                    if valid.any(): MEANCAM[(a, src, tk, l, g)] = cams[valid].mean(0)
                    row.update(edge=ef.mean() if len(ef) else np.nan, bottom=bf.mean() if len(bf) else np.nan,
                               edge_sem=ef.std(ddof=1) / np.sqrt(len(ef)) if len(ef) > 1 else np.nan,
                               bottom_sem=bf.std(ddof=1) / np.sqrt(len(bf)) if len(bf) > 1 else np.nan, empty=int((~valid).sum()))
                    stat_rows.append(row)
                    for c_ in range(min(KSHOW, n)):
                        axs[r_, c_].imshow(get_img(tname, pk[c_]), cmap="gray"); axs[r_, c_].imshow(cams[c_], cmap="jet", alpha=0.4)
                        axs[r_, c_].set_title(f"p = {P[pk[c_], j]:.2f}", fontsize=5.5, pad=1)
                    for c_, i in enumerate(pk):
                        pick_rows.append(dict(arch=a, src=src, test=tname, label=l, group=g, test_index=int(i), prob=float(P[i, j]), shown=c_ < KSHOW))
                fig.suptitle(f"{a}, trained on {trained}, tested on {tname} ({'in-hospital' if tk == 'in' else 'external'}): {l}  [threshold {thr0[j]:.2f}]", fontsize=7)
                fig.tight_layout()
                base = rf"{GD}\{SHORT[a]}\{trained}_to_{tname}_{l.replace(' ', '_')}"
                fig.savefig(base + ".pdf", bbox_inches="tight"); fig.savefig(base + ".png", dpi=300, bbox_inches="tight")
                if l in SHOW_LABELS: plt.show()
                plt.close(fig)
        del model; gc.collect(); torch.cuda.empty_cache(); print("done:", a, "trained on", trained)

ST = pd.DataFrame(stat_rows); ST.to_csv(rf"{GD}\gradcam_region_stats.csv", index=False)
pd.DataFrame(pick_rows).to_csv(rf"{GD}\gradcam_picked_images.csv", index=False)       # exactly which images were used
pickle.dump(RAW, open(rf"{GD}\gradcam_raw_per_image.pkl", "wb"))
np.savez_compressed(rf"{GD}\gradcam_mean_maps.npz", **{"|".join(k): v for k, v in MEANCAM.items()})
print("saved", len(ST), "stat rows; figures in", GD)

In [ ]:
def med(x): return float(np.median(x)) if len(x) else np.nan
def mwu(x, y): return float(mannwhitneyu(x, y).pvalue) if len(x) > 2 and len(y) > 2 else np.nan

rows = []
for a in ARCH:
    for src in ["nih", "cx"]:
        for l in LABELS:
            for mi, mn in enumerate(["edge", "bottom"]):
                for tk in ["in", "ext"]:                                   # correct vs missed positives
                    x, y = RAW[(a, src, tk, l, "TP")][mi], RAW[(a, src, tk, l, "FN")][mi]
                    rows.append(dict(arch=a, trained_on=src, test=tk, label=l, metric=mn, comparison="TP vs FN",
                                     median_A=med(x), median_B=med(y), p=mwu(x, y)))
                x, y = RAW[(a, src, "in", l, "TP")][mi], RAW[(a, src, "ext", l, "TP")][mi]   # same correct cases, home vs away hospital
                rows.append(dict(arch=a, trained_on=src, test="in vs ext", label=l, metric=mn, comparison="TP in-hospital vs TP external",
                                 median_A=med(x), median_B=med(y), p=mwu(x, y)))
TT = pd.DataFrame(rows); nt = TT.p.notna().sum()
TT["p_bonferroni"] = np.minimum(TT.p * nt, 1.0)
TT.to_csv(rf"{GD}\gradcam_tests.csv", index=False)
print(f"{nt} tests; significant after Bonferroni (p < 0.05): {(TT.p_bonferroni < 0.05).sum()}")
display(TT[TT.p_bonferroni < 0.05].sort_values("p").round(4).head(30))

# label-averaged region table (what to quote in the paper)
agg = ST.groupby(["arch", "trained_on", "tested_on", "group"])[["edge", "bottom"]].mean().unstack("group").round(3)
print(f"Uniform-heat baselines: edge {EDGE.mean():.2f}, bottom third {BOTTOM.mean():.2f}")
display(agg); agg.to_csv(rf"{GD}\gradcam_region_summary.csv")

In [ ]:
# Fig 11A: average heatmap per group (one figure per model and test set; rows = findings)
for a in ARCH:
    for src in ["nih", "cx"]:
        trained, away = ("NIH", "CheXpert") if src == "nih" else ("CheXpert", "NIH")
        for tk, tname in [("in", trained), ("ext", away)]:
            vmax = max([MEANCAM[k].max() for k in MEANCAM if k[:3] == (a, src, tk)] + [1e-3])
            fig, axs = plt.subplots(L, 4, figsize=(COL2, 8.6))
            for r_, l in enumerate(LABELS):
                for c_, g in enumerate(GROUPS):
                    ax_ = axs[r_, c_]; ax_.axis("off")
                    ax_.imshow(MEANCAM.get((a, src, tk, l, g), np.zeros((224, 224))), cmap="jet", vmin=0, vmax=vmax)
                    if r_ == 0: ax_.set_title(GNAME[g], fontsize=6)
                    if c_ == 0: ax_.text(-0.04, 0.5, l, transform=ax_.transAxes, rotation=90, va="center", ha="right", fontsize=6)
            fig.suptitle(f"Average Grad-CAM, {a}, trained on {trained}, tested on {tname}", fontsize=7); fig.tight_layout()
            base = rf"{GD}\{SHORT[a]}\MEAN_{trained}_to_{tname}"
            fig.savefig(base + ".pdf", bbox_inches="tight"); fig.savefig(base + ".png", dpi=300, bbox_inches="tight")
            if a == "DenseNet121" and src == "nih" and tk == "ext": plt.show()
            plt.close(fig)

# Fig 11B: where does the heat sit? edge / bottom fraction by group, 4 directions, both architectures
ST["direction"] = ST.trained_on + "→" + ST.tested_on
A2 = ST.groupby(["arch", "direction", "group"])[["edge", "bottom"]].mean().reset_index()
DIRS4 = ["CheXpert→CheXpert", "CheXpert→NIH", "NIH→NIH", "NIH→CheXpert"]
fig, axs = plt.subplots(2, 2, figsize=(COL2, 4.8), sharex=True)
for r_, a in enumerate(ARCH):
    for c_, (met, base_v, ttl) in enumerate([("edge", EDGE.mean(), "Heat in the outer 15% frame"), ("bottom", BOTTOM.mean(), "Heat in the bottom third")]):
        ax_ = axs[r_, c_]
        for k, d in enumerate(DIRS4):
            v = [A2[(A2.arch == a) & (A2.direction == d) & (A2.group == g)][met].mean() for g in GROUPS]
            ax_.bar(np.arange(4) + (k - 1.5) * 0.2, v, 0.2, color=["0.2", "0.45", "0.7", "0.9"][k], edgecolor="k", lw=0.4, label=d)
        ax_.axhline(base_v, color="k", ls=":", lw=0.7)
        ax_.set_xticks(range(4)); ax_.set_xticklabels(GROUPS); ax_.set(title=f"{a}: {ttl}", ylabel="Fraction of heat"); ax_.grid(True, axis="y", alpha=0.3)
axs[0, 0].legend(frameon=False, fontsize=5.5, ncol=2); fig.tight_layout()
fig.savefig(rf"{GD}\fig_gradcam_region_summary.pdf", bbox_inches="tight"); fig.savefig(rf"{GD}\fig_gradcam_region_summary.png", dpi=600, bbox_inches="tight"); plt.show()

In [ ]:
INCLUDE_WEIGHTS = False        # True adds all ConvNeXt .pt checkpoints (roughly 0.7 GB)
BUN = rf"{OUT}\PART34_bundle"; os.makedirs(BUN, exist_ok=True)
ign = shutil.ignore_patterns("*_last.pt", "*.u8", "__pycache__") if INCLUDE_WEIGHTS else shutil.ignore_patterns("*.pt", "*.u8", "__pycache__")
shutil.copytree(XD, rf"{BUN}\part34", dirs_exist_ok=True, ignore=ign)     # Cells 0-8 tables/figures/predictions + Grad-CAM
for f_ in [r"G:\chexpert_final_clean.csv", r"G:\nih_split.csv"]:
    if os.path.exists(f_): shutil.copy(f_, BUN)

json.dump(dict(date=str(datetime.date.today()), archs=ARCH, epochs=EPOCHS, seeds=SEEDS, batch_size=BS, lr=LR, image_size=224,
               labels=LABELS, recal_ks=KS, recal_draws=DRAWS, bootstrap=B, gradcam_seed=GSEED, gradcam_images_per_group=NSTAT,
               gradcam_model_seed=0, torch=torch.__version__, timm=timm.__version__), open(rf"{BUN}\settings_part34.json", "w"), indent=2)
open(rf"{BUN}\README_part34.txt", "w").write(
    "part34/                          ConvNeXt-T predictions (*_preds.npz), tables (table_*.csv), figures (fig_*.pdf/png), recal_raw.csv\n"
    "part34/gradcam/densenet121|convnext_tiny/   one figure per model, training set, test set and finding (TP/FN/FP/TN examples) + MEAN_* average maps\n"
    "part34/gradcam/gradcam_region_stats.csv     edge and bottom-third heat fractions per group\n"
    "part34/gradcam/gradcam_tests.csv            Mann-Whitney tests with Bonferroni correction\n"
    "part34/gradcam/gradcam_picked_images.csv    exact test-image indices used\n"
    "settings_part34.json                        all settings and library versions\n")
zp = shutil.make_archive(rf"{OUT}\PART34_bundle", "zip", BUN)
print("bundle:", BUN); print("zip:", zp, f"({os.path.getsize(zp) / 1e6:.0f} MB)")
for e in [".csv", ".pdf", ".png", ".npz", ".pkl", ".pt"]:
    print(e, sum(f.endswith(e) for _, _, fs in os.walk(BUN) for f in fs))